In [16]:
# Uber Fare Prediction (Minimal + Proper)
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler


In [17]:
# Load dataset
df = pd.read_csv("./uber.csv")

In [18]:
# Preprocessing
df = df.dropna(subset=['dropoff_latitude','dropoff_longitude'])
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')
df = df[(df['fare_amount'] > 0) & (df['passenger_count'] > 0)]


In [19]:
# Remove invalid coordinates
df = df[(df['pickup_longitude'].between(-80,-70)) & (df['dropoff_longitude'].between(-80,-70)) &
        (df['pickup_latitude'].between(35,45)) & (df['dropoff_latitude'].between(35,45))]


In [20]:
# Haversine distance
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    return R * 2 * np.arcsin(np.sqrt(np.sin((lat2-lat1)/2)**2 + 
           np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2))
df['distance_km'] = haversine(df['pickup_latitude'], df['pickup_longitude'],
                              df['dropoff_latitude'], df['dropoff_longitude'])
df = df[(df['distance_km'] > 0) & (df['distance_km'] < 100)]


In [21]:
# Extract datetime features
df['hour'] = df['pickup_datetime'].dt.hour
df['dayofweek'] = df['pickup_datetime'].dt.dayofweek
df['month'] = df['pickup_datetime'].dt.month


In [22]:
# Features and target
X = df[['distance_km', 'passenger_count', 'hour', 'dayofweek', 'month']]
y = df['fare_amount']

In [23]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [24]:
# Linear Regression
scaler = StandardScaler()
lr = LinearRegression().fit(scaler.fit_transform(X_train), y_train)
pred_lr = lr.predict(scaler.transform(X_test))


In [28]:
# Random Forest
rf = RandomForestRegressor(n_estimators=50, random_state=42, max_depth=15)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
# Evaluation
def metrics(y_true, y_pred): 
    return round(r2_score(y_true, y_pred),3), round(mean_squared_error(y_true, y_pred, squared=False),3), round(mean_absolute_error(y_true, y_pred),3)

print("Linear Regression (R2, RMSE, MAE):", metrics(y_test, pred_lr))
print("Random Forest (R2, RMSE, MAE):", metrics(y_test, pred_rf))


TypeError: got an unexpected keyword argument 'squared'